In [1]:
import dask.dataframe as dd

train = dd.read_parquet("D:/Projects/credit_risk_scoring/data/features/train_data.parquet")
sample = train.head(2000)

# Find constant columns (excluding identifiers and target)
exclude = ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD', 'target']
constant_cols = []

for col in train.columns:
    if col in exclude:
        continue
    if col in sample.columns:
        unique_count = sample[col].nunique()
        if unique_count <= 1:
            constant_cols.append(col)
            print(f"  {col}: only {unique_count} unique value(s)")

print(f"\nFound {len(constant_cols)} constant columns:")
print(constant_cols)

  DEFECT_SETTLEMENT_DATE: only 0 unique value(s)
  MODIFICATION_FLAG: only 0 unique value(s)
  ZERO_BALANCE_CODE: only 0 unique value(s)
  ZERO_BALANCE_EFFECTIVE_DATE: only 0 unique value(s)
  CURRENT_NON_INTEREST_BEARING_UPB: only 1 unique value(s)
  DDLPI: only 0 unique value(s)
  MI_RECOVERIES: only 0 unique value(s)
  NET_SALE_PROCEEDS: only 0 unique value(s)
  NON_MI_RECOVERIES: only 0 unique value(s)
  TOTAL_EXPENSES: only 0 unique value(s)
  LEGAL_COSTS: only 0 unique value(s)
  MAINTENANCE_AND_PRESERVATION_COSTS: only 0 unique value(s)
  TAXES_AND_INSURANCE: only 0 unique value(s)
  MISCELLANEOUS_EXPENSES: only 0 unique value(s)
  ACTUAL_LOSS_CALCULATION: only 0 unique value(s)
  CUMULATIVE_MODIFICATION_COST: only 0 unique value(s)
  INTEREST_RATE_STEP_INDICATOR: only 0 unique value(s)
  PAYMENT_DEFERRAL_FLAG: only 0 unique value(s)
  ELTV: only 0 unique value(s)
  ZERO_BALANCE_REMOVAL_UPB: only 0 unique value(s)
  DELINQUENT_ACCRUED_INTEREST: only 0 unique value(s)
  DELINQUEN

In [2]:

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Spark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window



In [3]:
# Set up display
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Create Spark session
spark = SparkSession.builder \
    .appName("SFLLD_Data_Quality") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "64") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("=" * 80)
print("FREDDIE MAC SFLLD - COMPREHENSIVE DATA QUALITY ANALYSIS")
print(f"Started at: {datetime.now()}")
print("=" * 80)


FREDDIE MAC SFLLD - COMPREHENSIVE DATA QUALITY ANALYSIS
Started at: 2026-07-12 03:59:57.235059


In [6]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================

DATA_PATH = "file:///D:/Projects/credit_risk_scoring/data/bronze/"

print("\n" + "=" * 80)
print("1. LOADING DATA")
print("=" * 80)

# Load bronze data
orig_df = spark.read.parquet(DATA_PATH + "origination_bronze.parquet")
perf_df = spark.read.parquet(DATA_PATH + "performance_bronze.parquet")

print(f"Origination records: {orig_df.count():,}")
print(f"Performance records: {perf_df.count():,}")
print(f"Origination columns: {len(orig_df.columns)}")
print(f"Performance columns: {len(perf_df.columns)}")



1. LOADING DATA
Origination records: 700,000
Performance records: 43,612,276
Origination columns: 34
Performance columns: 35


In [7]:
# =============================================================================
# 2. SCHEMA ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("2. SCHEMA ANALYSIS")
print("=" * 80)

print("\n--- Origination Schema ---")
orig_schema = []
for field in orig_df.schema.fields:
    null_count = orig_df.filter(col(field.name).isNull()).count()
    non_null_count = orig_df.count() - null_count
    null_pct = (null_count / orig_df.count()) * 100
    unique_count = orig_df.select(field.name).distinct().count()
    
    orig_schema.append({
        'column': field.name,
        'data_type': str(field.dataType),
        'null_count': null_count,
        'null_pct': null_pct,
        'unique_values': unique_count,
        'is_constant': unique_count <= 1
    })

orig_schema_df = pd.DataFrame(orig_schema)
print(orig_schema_df.to_string())

print("\n--- Performance Schema ---")
perf_schema = []
for field in perf_df.schema.fields:
    null_count = perf_df.filter(col(field.name).isNull()).count()
    non_null_count = perf_df.count() - null_count
    null_pct = (null_count / perf_df.count()) * 100
    unique_count = perf_df.select(field.name).distinct().count()
    
    perf_schema.append({
        'column': field.name,
        'data_type': str(field.dataType),
        'null_count': null_count,
        'null_pct': null_pct,
        'unique_values': unique_count,
        'is_constant': unique_count <= 1
    })

perf_schema_df = pd.DataFrame(perf_schema)
print(perf_schema_df.to_string())



2. SCHEMA ANALYSIS

--- Origination Schema ---
                         column        data_type  null_count  null_pct  unique_values  is_constant
0                  CREDIT_SCORE    IntegerType()           0      0.00            402        False
1            FIRST_PAYMENT_DATE     StringType()           0      0.00            186        False
2     FIRST_TIME_HOMEBUYER_FLAG     StringType()           0      0.00              3        False
3                 MATURITY_DATE     StringType()           0      0.00            495        False
4                           MSA     StringType()      213972     30.57            459        False
5                 MI_PERCENTAGE    IntegerType()           0      0.00             43        False
6               NUMBER_OF_UNITS    IntegerType()           0      0.00              5        False
7              OCCUPANCY_STATUS     StringType()           0      0.00              3        False
8                 ORIGINAL_CLTV    IntegerType()           0 

In [8]:
# =============================================================================
# 3. CONSTANT / NEAR-CONSTANT COLUMNS
# =============================================================================

print("\n" + "=" * 80)
print("3. CONSTANT / NEAR-CONSTANT COLUMNS")
print("=" * 80)

def find_constant_columns(df, df_name, threshold=0.99):
    """Find columns with dominant value share > threshold"""
    constant_cols = []
    near_constant_cols = []
    
    for col_name in df.columns:
        try:
            # Get value counts as percentage
            counts = df.groupBy(col_name).count().withColumn(
                'pct', col('count') / df.count()
            ).orderBy(desc('pct'))
            
            result = counts.first()
            if result:
                top_pct = float(result['pct'])
                if top_pct == 1.0:
                    constant_cols.append(col_name)
                elif top_pct >= threshold:
                    near_constant_cols.append((col_name, top_pct))
        except:
            pass
    
    print(f"\n{df_name} - Exactly Constant Columns ({len(constant_cols)}):")
    for col in sorted(constant_cols):
        print(f"  {col}")
    
    print(f"\n{df_name} - Near-Constant Columns (>{threshold*100:.0f}% same value):")
    for col, pct in sorted(near_constant_cols, key=lambda x: -x[1]):
        print(f"  {col}: {pct*100:.1f}%")

find_constant_columns(orig_df, "Origination")
find_constant_columns(perf_df, "Performance")



3. CONSTANT / NEAR-CONSTANT COLUMNS

Origination - Exactly Constant Columns (0):

Origination - Near-Constant Columns (>99% same value):

Performance - Exactly Constant Columns (0):

Performance - Near-Constant Columns (>99% same value):


In [9]:
# =============================================================================
# 4. MISSING DATA ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("4. MISSING DATA ANALYSIS")
print("=" * 80)

def analyze_missing_data(df, df_name, top_n=20):
    """Analyze missing data patterns"""
    
    # Get null counts per column
    null_counts = []
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        null_pct = (null_count / df.count()) * 100
        null_counts.append((col_name, null_count, null_pct))
    
    null_df = pd.DataFrame(null_counts, columns=['column', 'null_count', 'null_pct'])
    null_df = null_df.sort_values('null_pct', ascending=False)
    
    print(f"\n{df_name} - Columns with Missing Data (Top {top_n}):")
    print(null_df.head(top_n).to_string())
    
    return null_df

orig_nulls = analyze_missing_data(orig_df, "Origination")
perf_nulls = analyze_missing_data(perf_df, "Performance")



4. MISSING DATA ANALYSIS

Origination - Columns with Missing Data (Top 20):
                        column  null_count  null_pct
25       SUPER_CONFORMING_FLAG      694975     99.28
26    PRE_RELIEF_REFINANCE_LSN      647103     92.44
28  RELIEF_REFINANCE_INDICATOR      647103     92.44
4                          MSA      213972     30.57
1           FIRST_PAYMENT_DATE           0      0.00
0                 CREDIT_SCORE           0      0.00
6              NUMBER_OF_UNITS           0      0.00
2    FIRST_TIME_HOMEBUYER_FLAG           0      0.00
3                MATURITY_DATE           0      0.00
5                MI_PERCENTAGE           0      0.00
10                ORIGINAL_UPB           0      0.00
11                ORIGINAL_LTV           0      0.00
12      ORIGINAL_INTEREST_RATE           0      0.00
13                     CHANNEL           0      0.00
14                    PPM_FLAG           0      0.00
7             OCCUPANCY_STATUS           0      0.00
8                ORIGI

In [10]:

# =============================================================================
# 5. CATEGORICAL COLUMN ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("5. CATEGORICAL COLUMN ANALYSIS")
print("=" * 80)

def analyze_categorical_columns(df, df_name):
    """Analyze categorical columns (string/object types)"""
    
    categorical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    
    print(f"\n{df_name} - Categorical Columns ({len(categorical_cols)}):")
    
    cat_analysis = []
    for col_name in categorical_cols:
        distinct_count = df.select(col_name).distinct().count()
        null_count = df.filter(col(col_name).isNull()).count()
        null_pct = (null_count / df.count()) * 100
        
        # Sample values
        sample_values = df.select(col_name).distinct().limit(10).toPandas()[col_name].tolist()
        
        cat_analysis.append({
            'column': col_name,
            'distinct_values': distinct_count,
            'null_pct': null_pct,
            'sample_values': sample_values[:5],
            'is_high_cardinality': distinct_count > 100,
            'is_low_cardinality': distinct_count <= 20
        })
    
    cat_df = pd.DataFrame(cat_analysis)
    
    print("\nHigh Cardinality (>= 100 values):")
    high_card = cat_df[cat_df['is_high_cardinality']]
    for _, row in high_card.iterrows():
        print(f"  {row['column']}: {row['distinct_values']} values")
    
    print("\nLow Cardinality (<= 20 values, good for categorical):")
    low_card = cat_df[cat_df['is_low_cardinality']]
    for _, row in low_card.iterrows():
        print(f"  {row['column']}: {row['distinct_values']} values - {row['sample_values']}")
    
    return cat_df

orig_cat = analyze_categorical_columns(orig_df, "Origination")
perf_cat = analyze_categorical_columns(perf_df, "Performance")



5. CATEGORICAL COLUMN ANALYSIS

Origination - Categorical Columns (21):

High Cardinality (>= 100 values):
  FIRST_PAYMENT_DATE: 186 values
  MATURITY_DATE: 495 values
  MSA: 459 values
  POSTAL_CODE: 892 values
  LOAN_SEQUENCE_NUMBER: 700000 values
  PRE_RELIEF_REFINANCE_LSN: 52898 values

Low Cardinality (<= 20 values, good for categorical):
  FIRST_TIME_HOMEBUYER_FLAG: 3 values - ['N', 'Y', '9']
  OCCUPANCY_STATUS: 3 values - ['P', 'S', 'I']
  CHANNEL: 5 values - ['B', 'C', 'R', 'T', '9']
  PPM_FLAG: 2 values - ['N', 'Y']
  AMORTIZATION_TYPE: 1 values - ['FRM']
  PROPERTY_TYPE: 6 values - ['CP', 'SF', 'PU', 'MH', 'CO']
  LOAN_PURPOSE: 4 values - ['N', 'C', 'P', '9']
  SUPER_CONFORMING_FLAG: 2 values - ['Y', None]
  SPECIAL_ELIGIBILITY_PROGRAM: 1 values - ['9']
  RELIEF_REFINANCE_INDICATOR: 2 values - ['Y', None]
  IO_INDICATOR: 1 values - ['N']
  MI_CANCELLATION_INDICATOR: 1 values - ['9']

Performance - Categorical Columns (12):

High Cardinality (>= 100 values):
  LOAN_SEQUENCE_N

In [11]:

# =============================================================================
# 6. NUMERICAL COLUMN ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("6. NUMERICAL COLUMN ANALYSIS")
print("=" * 80)

def analyze_numeric_columns(df, df_name):
    """Analyze numeric columns for outliers, skewness, etc."""
    
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, FloatType, LongType))]
    
    print(f"\n{df_name} - Numerical Columns ({len(numeric_cols)}):")
    
    num_analysis = []
    for col_name in numeric_cols:
        try:
            stats = df.select(
                min(col_name).alias('min'),
                max(col_name).alias('max'),
                mean(col_name).alias('mean'),
                stddev(col_name).alias('std'),
                count(col_name).alias('non_null'),
                approx_count_distinct(col_name).alias('unique')
            ).collect()[0]
            
            # Detect zero-variance columns
            is_constant = stats['min'] == stats['max'] if stats['min'] is not None and stats['max'] is not None else False
            is_near_zero_variance = (stats['std'] or 0) < 0.01
            
            num_analysis.append({
                'column': col_name,
                'min': stats['min'],
                'max': stats['max'],
                'mean': stats['mean'],
                'std': stats['std'],
                'non_null': stats['non_null'],
                'unique': stats['unique'],
                'is_constant': is_constant,
                'is_near_zero_variance': is_near_zero_variance
            })
        except:
            num_analysis.append({'column': col_name, 'error': 'Could not compute'})
    
    num_df = pd.DataFrame(num_analysis)
    
    # Show constant columns
    constant_cols = num_df[num_df['is_constant'] == True]
    if len(constant_cols) > 0:
        print(f"\nConstant columns ({len(constant_cols)}):")
        print(constant_cols[['column', 'min', 'max']].to_string())
    
    # Show zero/near-zero variance
    near_zero = num_df[num_df['is_near_zero_variance'] == True]
    if len(near_zero) > 0:
        print(f"\nNear-zero variance columns ({len(near_zero)}):")
        print(near_zero[['column', 'mean', 'std']].to_string())
    
    return num_df

orig_num = analyze_numeric_columns(orig_df, "Origination")
perf_num = analyze_numeric_columns(perf_df, "Performance")



6. NUMERICAL COLUMN ANALYSIS

Origination - Numerical Columns (12):

Constant columns (1):
                       column  min  max
10  PROPERTY_VALUATION_METHOD 7.00 7.00

Near-zero variance columns (1):
                       column  mean  std
10  PROPERTY_VALUATION_METHOD  7.00 0.00

Performance - Numerical Columns (22):


In [12]:

# =============================================================================
# 7. DATE/TIME COLUMN ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("7. DATE/TIME COLUMN ANALYSIS")
print("=" * 80)

def analyze_date_columns(df, df_name):
    """Analyze date columns for validity and range"""
    
    date_cols = [f.name for f in df.schema.fields if 'DATE' in f.name.upper() or 'PERIOD' in f.name.upper()]
    
    print(f"\n{df_name} - Date/Period Columns ({len(date_cols)}):")
    
    for col_name in date_cols:
        try:
            # Check format (should be YYYYMM)
            sample = df.select(col_name).filter(col(col_name).isNotNull()).limit(5).collect()
            if sample:
                print(f"\n  {col_name}:")
                print(f"    Sample: {[row[col_name] for row in sample]}")
                
                # Check for invalid formats
                invalid = df.filter(length(col(col_name)) != 6).count()
                print(f"    Invalid format (not 6 chars): {invalid}")
                
                # Check range
                if invalid == 0:
                    min_val = df.select(min(col(col_name))).collect()[0][0]
                    max_val = df.select(max(col(col_name))).collect()[0][0]
                    print(f"    Range: {min_val} to {max_val}")
        except Exception as e:
            print(f"  Error analyzing {col_name}: {e}")

analyze_date_columns(orig_df, "Origination")
analyze_date_columns(perf_df, "Performance")



7. DATE/TIME COLUMN ANALYSIS

Origination - Date/Period Columns (2):

  FIRST_PAYMENT_DATE:
    Sample: ['201203', '201203', '201203', '201203', '201203']
    Invalid format (not 6 chars): 0
    Range: 199902 to 201702

  MATURITY_DATE:
    Sample: ['202702', '204202', '204202', '202702', '203202']
    Invalid format (not 6 chars): 0
    Range: 200308 to 205301

Performance - Date/Period Columns (3):

  MONTHLY_REPORTING_PERIOD:
    Sample: ['200101', '200102', '200103', '200104', '200105']
    Invalid format (not 6 chars): 0
    Range: 199901 to 202509

  DEFECT_SETTLEMENT_DATE:
    Sample: ['200108', '200106', '200111', '200101', '200102']
    Invalid format (not 6 chars): 0
    Range: 199902 to 202409

  ZERO_BALANCE_EFFECTIVE_DATE:
    Sample: ['200111', '200103', '200111', '200110', '200104']
    Invalid format (not 6 chars): 0
    Range: 199902 to 202509


In [13]:

# =============================================================================
# 8. LOAN AGE / REPORTING PERIOD CONSISTENCY
# =============================================================================

print("\n" + "=" * 80)
print("8. LOAN AGE / REPORTING PERIOD CONSISTENCY")
print("=" * 80)

def check_loan_age_consistency(df):
    """Check that LOAN_AGE matches reporting period minus origination date"""
    
    print("\nChecking LOAN_AGE consistency...")
    
    # Get a sample of loans
    sample_loans = df.select("LOAN_SEQUENCE_NUMBER").distinct().limit(1000).collect()
    sample_ids = [row[0] for row in sample_loans]
    
    # Analyze a subset
    df_sample = df.filter(col("LOAN_SEQUENCE_NUMBER").isin(sample_ids))
    
    # Check for negative or impossible loan ages
    negative_ages = df_sample.filter(col("LOAN_AGE") < 0).count()
    zero_ages = df_sample.filter(col("LOAN_AGE") == 0).count()
    max_age = df_sample.select(max("LOAN_AGE")).collect()[0][0]
    
    print(f"  Negative loan ages: {negative_ages}")
    print(f"  Zero loan ages: {zero_ages}")
    print(f"  Maximum loan age: {max_age}")
    
    # Check LOAN_AGE vs REMAINING_MONTHS_TO_LEGAL_MATURITY
    inconsistent = df_sample.filter(
        (col("LOAN_AGE") + col("REMAINING_MONTHS_TO_LEGAL_MATURITY")) != col("ORIGINAL_LOAN_TERM")
    ).count()
    print(f"  Inconsistent age + remaining != original term: {inconsistent}")

check_loan_age_consistency(perf_df)



8. LOAN AGE / REPORTING PERIOD CONSISTENCY

Checking LOAN_AGE consistency...
  Negative loan ages: 0
  Zero loan ages: 796
  Maximum loan age: 302


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ORIGINAL_LOAN_TERM` cannot be resolved. Did you mean one of the following? [`LEGAL_COSTS`, `LOAN_AGE`, `MODIFICATION_FLAG`, `TOTAL_EXPENSES`, `MI_RECOVERIES`].;
'Filter NOT ((LOAN_AGE#72 + REMAINING_MONTHS_TO_LEGAL_MATURITY#73) = 'ORIGINAL_LOAN_TERM)
+- Filter LOAN_SEQUENCE_NUMBER#68 IN (F00Q30086332,F00Q30087382,F00Q30087845,F00Q30088469,F00Q30090412,F00Q30094170,F00Q30094315,F00Q30097259,F00Q30099574,F00Q30104438,F00Q30104938,F00Q30111731,F00Q30111978,F00Q30113025,F00Q30113718,F00Q30116392,F00Q30123130,F00Q30126892,F00Q30128321,F00Q30128448,F00Q30134893,F00Q30137651,F00Q30139643,F00Q30145088,F00Q30148679,F00Q30150232,F00Q30153217,F00Q30154443,F00Q30154701,F00Q30156724,F00Q30158384,F00Q30159424,F00Q30159594,F00Q30160844,F00Q30161202,F00Q30162139,F00Q30162491,F00Q30168298,F00Q30169942,F00Q30174625,F00Q30175239,F00Q30175636,F00Q30176603,F00Q30176636,F00Q30177027,F00Q30177621,F00Q30177722,F00

In [ ]:

# =============================================================================
# 9. DELINQUENCY STATUS ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("9. DELINQUENCY STATUS ANALYSIS")
print("=" * 80)

def analyze_delinquency(df):
    """Analyze delinquency status distribution and transitions"""
    
    print("\nDelinquency Status Distribution:")
    status_dist = df.groupBy("CURRENT_LOAN_DELINQUENCY_STATUS").count().orderBy("CURRENT_LOAN_DELINQUENCY_STATUS")
    status_dist_pd = status_dist.toPandas()
    status_dist_pd['pct'] = (status_dist_pd['count'] / df.count()) * 100
    print(status_dist_pd.to_string())
    
    # Check for invalid status codes
    valid_statuses = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'RA']
    invalid = df.filter(~col("CURRENT_LOAN_DELINQUENCY_STATUS").isin(valid_statuses)).count()
    print(f"\nInvalid delinquency status codes: {invalid}")
    
    # Analyze transitions (look at a sample)
    print("\nAnalyzing delinquency transitions (sample)...")
    window_spec = Window.partitionBy("LOAN_SEQUENCE_NUMBER").orderBy("MONTHLY_REPORTING_PERIOD")
    
    df_with_lag = df.withColumn(
        "prev_status", 
        lag("CURRENT_LOAN_DELINQUENCY_STATUS").over(window_spec)
    )
    
    transitions = df_with_lag.filter(
        col("prev_status").isNotNull() & 
        (col("CURRENT_LOAN_DELINQUENCY_STATUS") != col("prev_status"))
    ).select("prev_status", "CURRENT_LOAN_DELINQUENCY_STATUS").limit(1000)
    
    print(f"  Sample transitions: {transitions.count()} recorded")

analyze_delinquency(perf_df)



9. DELINQUENCY STATUS ANALYSIS

Delinquency Status Distribution:
    CURRENT_LOAN_DELINQUENCY_STATUS     count   pct
0                                 0  41982165 96.26
1                                 1    661296  1.52
2                                10     27072  0.06
3                               100        53  0.00
4                               101        46  0.00
5                               102        46  0.00
6                               103        42  0.00
7                               104        43  0.00
8                               105        39  0.00
9                               106        41  0.00
10                              107        40  0.00
11                              108        38  0.00
12                              109        38  0.00
13                               11     23904  0.05
14                              110        36  0.00
15                              111        34  0.00
16                              112        31  0.0

In [ ]:

# =============================================================================
# 10. TERMINATION / ZERO BALANCE ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("10. TERMINATION / ZERO BALANCE ANALYSIS")
print("=" * 80)

def analyze_termination(df):
    """Analyze loan termination patterns"""
    
    print("\nZero Balance Code Distribution:")
    zb_dist = df.groupBy("ZERO_BALANCE_CODE").count().orderBy("ZERO_BALANCE_CODE")
    zb_dist_pd = zb_dist.toPandas()
    zb_dist_pd['pct'] = (zb_dist_pd['count'] / df.count()) * 100
    print(zb_dist_pd.to_string())
    
    print("\nTermination Analysis:")
    terminated = df.filter(col("ZERO_BALANCE_CODE").isNotNull())
    active = df.filter(col("ZERO_BALANCE_CODE").isNull())
    
    print(f"  Active loans: {active.count():,}")
    print(f"  Terminated loans: {terminated.count():,}")
    print(f"  Termination rate: {(terminated.count() / df.count()) * 100:.2f}%")
    
    # Check if loan can re-enter after termination
    re_entered = df.groupBy("LOAN_SEQUENCE_NUMBER").agg(
        count("ZERO_BALANCE_CODE").alias("termination_events")
    ).filter(col("termination_events") > 1).count()
    
    print(f"  Loans with multiple termination events: {re_entered}")

analyze_termination(perf_df)


In [ ]:

# =============================================================================
# 11. MODIFICATION ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("11. LOAN MODIFICATION ANALYSIS")
print("=" * 80)

def analyze_modifications(df):
    """Analyze loan modification patterns"""
    
    print("\nModification Flag Distribution:")
    mod_dist = df.groupBy("MODIFICATION_FLAG").count().orderBy("MODIFICATION_FLAG")
    mod_dist_pd = mod_dist.toPandas()
    mod_dist_pd['pct'] = (mod_dist_pd['count'] / df.count()) * 100
    print(mod_dist_pd.to_string())
    
    # Loans with modifications
    mod_loans = df.filter(col("MODIFICATION_FLAG") == "Y")
    print(f"\nLoans with modifications: {mod_loans.count():,}")
    print(f"  Percentage: {(mod_loans.count() / df.count()) * 100:.2f}%")
    
    # Multiple modifications
    mod_counts = df.filter(col("MODIFICATION_FLAG") == "Y").groupBy("LOAN_SEQUENCE_NUMBER").count()
    multi_mod = mod_counts.filter(col("count") > 1).count()
    print(f"  Loans with multiple modifications: {multi_mod}")

analyze_modifications(perf_df)


In [ ]:

# =============================================================================
# 12. DATA QUALITY SCORECARD
# =============================================================================

print("\n" + "=" * 80)
print("12. DATA QUALITY SCORECARD - SUMMARY")
print("=" * 80)

def generate_quality_scorecard():
    """Generate a comprehensive data quality scorecard"""
    
    scorecard = {
        'dataset': [],
        'metric': [],
        'value': [],
        'status': []
    }
    
    # Overall dataset metrics
    scorecard['dataset'].append('Origination')
    scorecard['metric'].append('Total Records')
    scorecard['value'].append(orig_df.count())
    scorecard['status'].append('OK' if orig_df.count() > 0 else 'CRITICAL')
    
    scorecard['dataset'].append('Performance')
    scorecard['metric'].append('Total Records')
    scorecard['value'].append(perf_df.count())
    scorecard['status'].append('OK' if perf_df.count() > 0 else 'CRITICAL')
    
    # Missing data rates
    for df, name in [(orig_df, 'Origination'), (perf_df, 'Performance')]:
        null_pcts = []
        for col_name in df.columns:
            null_count = df.filter(col(col_name).isNull()).count()
            null_pct = (null_count / df.count()) * 100 if df.count() > 0 else 0
            null_pcts.append(null_pct)
        
        avg_null = np.mean(null_pcts)
        max_null = np.max(null_pcts)
        
        scorecard['dataset'].append(name)
        scorecard['metric'].append('Average Null Rate')
        scorecard['value'].append(f'{avg_null:.1f}%')
        scorecard['status'].append('OK' if avg_null < 20 else 'WARNING')
        
        scorecard['dataset'].append(name)
        scorecard['metric'].append('Max Null Rate')
        scorecard['value'].append(f'{max_null:.1f}%')
        scorecard['status'].append('OK' if max_null < 50 else 'WARNING')
    
    # Constant columns
    for df, name in [(orig_df, 'Origination'), (perf_df, 'Performance')]:
        constant_count = 0
        for col_name in df.columns:
            unique_count = df.select(col_name).distinct().count()
            if unique_count <= 1:
                constant_count += 1
        
        scorecard['dataset'].append(name)
        scorecard['metric'].append('Constant Columns')
        scorecard['value'].append(constant_count)
        scorecard['status'].append('OK' if constant_count < 5 else 'WARNING')
    
    # Delinquency coverage
    has_delinquency = perf_df.filter(col("CURRENT_LOAN_DELINQUENCY_STATUS") != "0").count()
    delinquency_rate = (has_delinquency / perf_df.count()) * 100 if perf_df.count() > 0 else 0
    
    scorecard['dataset'].append('Performance')
    scorecard['metric'].append('Delinquency Rate')
    scorecard['value'].append(f'{delinquency_rate:.2f}%')
    scorecard['status'].append('OK' if delinquency_rate > 1 else 'WARNING')
    
    return pd.DataFrame(scorecard)

quality_scorecard = generate_quality_scorecard()
print(quality_scorecard.to_string())


In [ ]:

# =============================================================================
# 13. JOIN ANALYSIS (Origination + Performance)
# =============================================================================

print("\n" + "=" * 80)
print("13. JOIN ANALYSIS - ORIGINATION + PERFORMANCE")
print("=" * 80)

def analyze_join():
    """Analyze the relationship between origination and performance data"""
    
    print("\nLoan Coverage:")
    orig_loans = orig_df.select("LOAN_SEQUENCE_NUMBER").distinct().count()
    perf_loans = perf_df.select("LOAN_SEQUENCE_NUMBER").distinct().count()
    
    print(f"  Loans in origination: {orig_loans:,}")
    print(f"  Loans in performance: {perf_loans:,}")
    
    # Find loans only in one dataset
    orig_set = set([row[0] for row in orig_df.select("LOAN_SEQUENCE_NUMBER").distinct().limit(10000).collect()])
    perf_set = set([row[0] for row in perf_df.select("LOAN_SEQUENCE_NUMBER").distinct().limit(10000).collect()])
    
    only_orig = orig_set - perf_set
    only_perf = perf_set - orig_set
    
    print(f"  Loans only in origination (sample): {len(only_orig)}")
    print(f"  Loans only in performance (sample): {len(only_perf)}")
    
    # Check join key formatting
    print("\nLoan ID Format Check:")
    orig_sample = orig_df.select("LOAN_SEQUENCE_NUMBER").limit(5).collect()
    perf_sample = perf_df.select("LOAN_SEQUENCE_NUMBER").limit(5).collect()
    
    print(f"  Origination sample: {[row[0] for row in orig_sample]}")
    print(f"  Performance sample: {[row[0] for row in perf_sample]}")

analyze_join()


In [ ]:

# =============================================================================
# 14. PER-LOAN PERFORMANCE ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("14. PER-LOAN PERFORMANCE STATISTICS")
print("=" * 80)

def analyze_per_loan_stats():
    """Analyze performance metrics per loan"""
    
    # Get per-loan summary
    loan_stats = perf_df.groupBy("LOAN_SEQUENCE_NUMBER").agg(
        count("*").alias("monthly_records"),
        min("MONTHLY_REPORTING_PERIOD").alias("first_reporting_date"),
        max("MONTHLY_REPORTING_PERIOD").alias("last_reporting_date"),
        max("LOAN_AGE").alias("max_age"),
        max("CURRENT_LOAN_DELINQUENCY_STATUS").alias("max_delinquency")
    )
    
    print("\nLoan Coverage (months per loan):")
    coverage_dist = loan_stats.groupBy("monthly_records").count().orderBy("monthly_records").limit(20)
    coverage_pd = coverage_dist.toPandas()
    print(coverage_pd.to_string())
    
    # Loans with very few records
    few_records = loan_stats.filter(col("monthly_records") < 6).count()
    print(f"\nLoans with <6 records: {few_records:,}")

analyze_per_loan_stats()


In [ ]:

# =============================================================================
# 15. VINTAGE YEAR ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("15. VINTAGE YEAR ANALYSIS")
print("=" * 80)

def analyze_vintages():
    """Analyze loans by vintage year"""
    
    print("\nOrigination by Year:")
    vintage_dist = orig_df.groupBy("vintage_year").count().orderBy("vintage_year")
    vintage_pd = vintage_dist.toPandas()
    vintage_pd['pct'] = (vintage_pd['count'] / vintage_pd['count'].sum()) * 100
    print(vintage_pd.to_string())
    
    # Check for missing years
    expected_years = list(range(1999, 2013))
    found_years = vintage_pd['vintage_year'].tolist()
    missing_years = [y for y in expected_years if y not in found_years]
    
    if missing_years:
        print(f"\nMissing vintage years: {missing_years}")

analyze_vintages()


In [ ]:

# =============================================================================
# 16. SANITY CHECKS
# =============================================================================

print("\n" + "=" * 80)
print("16. SANITY CHECKS")
print("=" * 80)

def run_sanity_checks():
    """Run various sanity checks on the data"""
    
    checks = []
    
    # 1. Check for negative balances
    neg_upb = perf_df.filter(col("CURRENT_ACTUAL_UPB") < 0).count()
    checks.append(("Negative UPB", neg_upb, "PASS" if neg_upb == 0 else "FAIL"))
    
    # 2. Check for interest rates > 30%
    high_rate = perf_df.filter(col("CURRENT_INTEREST_RATE") > 30).count()
    checks.append(("Interest rate > 30%", high_rate, "PASS" if high_rate == 0 else "WARNING"))
    
    # 3. Check for impossible loan ages (> 480 months)
    old_loans = perf_df.filter(col("LOAN_AGE") > 480).count()
    checks.append(("Loan age > 480 months", old_loans, "PASS" if old_loans == 0 else "FAIL"))
    
    # 4. Check for LTV > 100% (unless HARP)
    high_ltv = orig_df.filter(col("ORIGINAL_LTV") > 100).count()
    checks.append(("LTV > 100%", high_ltv, "PASS" if high_ltv == 0 else "WARNING"))
    
    # 5. Check for DTI > 65%
    high_dti = orig_df.filter(col("ORIGINAL_DTI") > 65).count()
    checks.append(("DTI > 65%", high_dti, "PASS" if high_dti == 0 else "WARNING"))
    
    # 6. Check for credit score outside 300-850
    bad_credit = orig_df.filter((col("CREDIT_SCORE") < 300) | (col("CREDIT_SCORE") > 850)).count()
    checks.append(("Credit score outside 300-850", bad_credit, "PASS" if bad_credit == 0 else "FAIL"))
    
    # 7. Check for same loan appearing in multiple vintages
    duplicate_loans = orig_df.groupBy("LOAN_SEQUENCE_NUMBER").count().filter(col("count") > 1).count()
    checks.append(("Duplicate loan IDs in origination", duplicate_loans, "PASS" if duplicate_loans == 0 else "FAIL"))
    
    # Print results
    print("\nSanity Check Results:")
    print("-" * 60)
    for check, value, status in checks:
        print(f"  {check:<40} {value:>10}  {status}")
    print("-" * 60)
    
    return checks

sanity_checks = run_sanity_checks()


In [ ]:

# =============================================================================
# 17. RECOMMENDATIONS
# =============================================================================

print("\n" + "=" * 80)
print("17. RECOMMENDATIONS FOR DATA CLEANING")
print("=" * 80)

def generate_recommendations():
    """Generate data cleaning recommendations based on findings"""
    
    recommendations = []
    
    # Check for high null rates
    high_null_cols = []
    for col_name in perf_df.columns:
        null_count = perf_df.filter(col(col_name).isNull()).count()
        null_pct = (null_count / perf_df.count()) * 100
        if null_pct > 90 and null_count > 0:
            high_null_cols.append((col_name, null_pct))
    
    if high_null_cols:
        recommendations.append({
            'priority': 'HIGH',
            'action': 'Drop columns with >90% null values',
            'columns': [c[0] for c in high_null_cols[:10]]
        })
    
    # Check for constant columns
    constant_cols = []
    for col_name in perf_df.columns:
        unique_count = perf_df.select(col_name).distinct().count()
        if unique_count <= 1:
            constant_cols.append(col_name)
    
    if constant_cols:
        recommendations.append({
            'priority': 'HIGH',
            'action': 'Drop constant columns (all same value)',
            'columns': constant_cols[:10]
        })
    
    # Check for invalid values
    recommendations.append({
        'priority': 'MEDIUM',
        'action': 'Clean numerical outliers',
        'details': 'Credit score (300-850), DTI (0-65), LTV (0-105), Interest rate (0-30)'
    })
    
    recommendations.append({
        'priority': 'MEDIUM',
        'action': 'Validate date formats',
        'details': 'All date columns should be YYYYMM format (6 digits)'
    })
    
    recommendations.append({
        'priority': 'LOW',
        'action': 'Impute missing values',
        'details': 'Use mode for categorical, median for numerical'
    })
    
    return pd.DataFrame(recommendations)

recommendations_df = generate_recommendations()
print(recommendations_df.to_string())


In [ ]:

# =============================================================================
# 18. SUMMARY REPORT
# =============================================================================

print("\n" + "=" * 80)
print("18. FINAL SUMMARY")
print("=" * 80)

print(f"""
DATA QUALITY SUMMARY:
=====================

ORIGINATION DATA:
  - Total records: {orig_df.count():,}
  - Total columns: {len(orig_df.columns)}
  - Columns with >90% nulls: {sum(1 for c in orig_df.columns if (orig_df.filter(col(c).isNull()).count() / orig_df.count()) > 0.9)}
  - Constant columns: {sum(1 for c in orig_df.columns if orig_df.select(c).distinct().count() <= 1)}
  - Year range: {orig_df.select(min('vintage_year')).collect()[0][0]} - {orig_df.select(max('vintage_year')).collect()[0][0]}

PERFORMANCE DATA:
  - Total records: {perf_df.count():,}
  - Total columns: {len(perf_df.columns)}
  - Columns with >90% nulls: {sum(1 for c in perf_df.columns if (perf_df.filter(col(c).isNull()).count() / perf_df.count()) > 0.9)}
  - Constant columns: {sum(1 for c in perf_df.columns if perf_df.select(c).distinct().count() <= 1)}
  - Reporting period range: {perf_df.select(min('MONTHLY_REPORTING_PERIOD')).collect()[0][0]} - {perf_df.select(max('MONTHLY_REPORTING_PERIOD')).collect()[0][0]}

JOIN QUALITY:
  - Loans matching: {perf_df.select('LOAN_SEQUENCE_NUMBER').distinct().join(orig_df.select('LOAN_SEQUENCE_NUMBER').distinct(), on='LOAN_SEQUENCE_NUMBER', how='inner').count():,}

RECOMMENDED ACTIONS:
  1. Drop {sum(1 for c in perf_df.columns if (perf_df.filter(col(c).isNull()).count() / perf_df.count()) > 0.9)} high-null columns
  2. Drop {sum(1 for c in orig_df.columns if orig_df.select(c).distinct().count() <= 1)} constant columns from origination
  3. Drop {sum(1 for c in perf_df.columns if perf_df.select(c).distinct().count() <= 1)} constant columns from performance
  4. Clean outliers in numerical columns
  5. Validate date formats
""")

print("=" * 80)
print(f"EDA COMPLETED AT: {datetime.now()}")
print("=" * 80)